# Shared Geometry as a Rosetta Stone: Cross-Modal Alignment Without Paired Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dominik-schnaus/unpaired-rosetta/blob/main/unpaired_rosetta.ipynb)

This notebook reproduces any unpaired or few-pair vision-language number of the paper and explains the method on the way.
It

1. lets you choose the datasets, models, number of known pairs and seed,
2. fetches the embeddings (after telling you how large they are),
3. builds the two unpaired training sets,
4. aligns them with a short implementation of our **Wasserstein Procrustes** aligner that follows Algorithms 1 and 2 of the paper line by line,
5. evaluates the alignment on the paired validation data and on zero-shot classification.

In [ ]:
# On Colab: download the code and install it. Locally (pixi install), nothing to do.
import os
import sys

if "google.colab" in sys.modules and not os.path.exists("unpaired-rosetta"):
    !git clone --quiet https://github.com/dominik-schnaus/unpaired-rosetta.git
    %cd unpaired-rosetta
    !pip install --quiet -e .
    os.environ["UNPAIRED_ROSETTA_ROOT"] = "/content/unpaired_rosetta_data"  # embeddings and raw data go here
os.environ.setdefault("UNPAIRED_ROSETTA_ROOT", "storage")
os.environ.setdefault("MKL_CBWR", "AVX2,STRICT")  # the same floating-point results on every x86 CPU
os.environ.setdefault("ATEN_CPU_CAPABILITY", "avx2")

In [ ]:
import pandas as pd
import torch
from ipywidgets import Dropdown, IntSlider, SelectMultiple, Text, ToggleButtons, VBox
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits

from modalities import image_captions
from modalities.common import download_precomputed, format_bytes
from unpaired_rosetta.embeddings import load_rows, num_rows
from unpaired_rosetta.qap import MPOptQAPSolver
from unpaired_rosetta.randomness import SEEDS, Randomness

## 1. Choose the setting

The training corpora provide the unpaired image and caption embeddings. Choosing a different corpus for the captions gives the cross-dataset setting of the paper (e.g. MS COCO images and SPC captions), where no image has its caption on the other side at all.
Validation always uses the paired MS COCO 2014 validation split. Known pairs are drawn from MS COCO train by default.
Alternatively, choose *own embeddings* and give the paths of your own `.pt` files (`[n, d]` tensors).

In [ ]:
dataset_names = {name: key for key, name in image_captions.TRAINING_DATASETS.items()}
vision_names = {name: key for key, name in image_captions.VISION_MODELS.items()}
language_names = {name: key for key, name in image_captions.LANGUAGE_MODELS.items()}

source = ToggleButtons(options=["paper embeddings", "own embeddings"], description="Embeddings")
images_from = Dropdown(options=list(dataset_names), value="MS COCO", description="Images from")
captions_from = Dropdown(options=list(dataset_names), value="MS COCO", description="Captions from")
vision_model = Dropdown(options=list(vision_names), value="DINOv2 B/14", description="Vision")
language_model = Dropdown(options=list(language_names), value="MPNet", description="Language")
classification = SelectMultiple(options=list(image_captions.CLASSIFICATION_DATASETS), value=["CIFAR-10"], description="Zero-shot")
num_pairs = Dropdown(options=[0, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000], value=0, description="Known pairs")
pairs_from = Dropdown(options=list(dataset_names), value="MS COCO", description="Pairs from")
seed = Dropdown(options=SEEDS, value=SEEDS[0], description="Seed")
own_x = Text(description="Own X", placeholder="/path/to/space_x.pt")
own_y = Text(description="Own Y", placeholder="/path/to/space_y.pt")
own_val_x = Text(description="Own val X", placeholder="paired validation, row i ↔ row i")
own_val_y = Text(description="Own val Y", placeholder="paired validation, row i ↔ row i")
VBox([source, images_from, captions_from, vision_model, language_model, classification, num_pairs, pairs_from, seed,
      own_x, own_y, own_val_x, own_val_y])

## 2. Embeddings

The cell lists the embedding files this setting needs and how large the missing ones are. It asks before downloading anything:
`download` fetches the precomputed embeddings of the paper from the Hugging Face Hub (the exact files behind its numbers),
`compute` downloads the raw images and captions and computes the embeddings with `modalities/image_captions/embed.py` (needs a GPU, and the Qwen3-8B models take hours).

In [ ]:
if source.value == "paper embeddings":
    dataset_x, dataset_y = dataset_names[images_from.value], dataset_names[captions_from.value]
    model_x, model_y = vision_names[vision_model.value], language_names[language_model.value]
    pairs_dataset = dataset_names[pairs_from.value] if num_pairs.value > 0 else None
    files = image_captions.files_of_setting(dataset_x, dataset_y, model_x, model_y, pairs_dataset, list(classification.value))
    missing = [file for file in files if not file.exists()]
    labels = [image_captions.label_file(dataset) for dataset in classification.value]
    missing_labels = [label for label in labels if not os.path.exists(os.path.join(os.environ["UNPAIRED_ROSETTA_ROOT"], label))]
    display(pd.DataFrame([{"file": file.relative_path, "size": format_bytes(file.num_bytes), "present": file.exists()} for file in files]))
    if missing or missing_labels:
        raw = {data.name: data for data in (image_captions.raw_data(file) for file in missing) if data is not None}
        print(f"Missing embeddings: {format_bytes(sum(file.num_bytes for file in missing))} to download and store.")
        print(f"Computing them instead downloads {format_bytes(sum(r.download_bytes for r in raw.values()))} of raw data "
              f"and stores {format_bytes(sum(r.stored_bytes for r in raw.values()))} ({', '.join(raw)}).")
        answer = input("Type 'download', 'compute', or anything else to stop: ").strip()
        if answer == "download":
            download_precomputed([file.relative_path for file in missing] + missing_labels)
        elif answer == "compute":
            from modalities.image_captions.embed import compute_embeddings
            compute_embeddings(missing)
        else:
            raise SystemExit("Stopped: the embeddings are missing.")

## 3. Training sets

For a paired corpus we draw one random permutation of its items and give the first half to the images and the second half to the captions.

In [ ]:
randomness = Randomness(seed.value)  # every random choice of this run comes from here


def disjoint_split(num_items):
    permutation = randomness.permutation(num_items)
    return permutation[: num_items // 2], permutation[num_items // 2 :]  # an odd item goes to the second half


if source.value == "own embeddings":
    X, Y = load_rows(own_x.value), load_rows(own_y.value)
    X_pairs = Y_pairs = None
else:
    path_x, path_y = image_captions.vision_file(dataset_x, model_x).path, image_captions.language_file(dataset_y, model_y).path
    if dataset_x == dataset_y:
        indices_x, indices_y = disjoint_split(num_rows(path_x))
        assert not set(indices_x.tolist()) & set(indices_y.tolist()), "the two training sets must not share an item"
    else:  # two different corpora: there is no pairing to begin with
        indices_x, indices_y = randomness.permutation(num_rows(path_x)), randomness.permutation(num_rows(path_y))
    X, Y = load_rows(path_x, indices_x), load_rows(path_y, indices_y)  # unit-length float32 embeddings
    X_pairs = Y_pairs = None
    if num_pairs.value > 0:
        pairs = randomness.subset(num_rows(image_captions.vision_file(pairs_dataset, model_x).path), num_pairs.value)
        X_pairs = load_rows(image_captions.vision_file(pairs_dataset, model_x).path, pairs)
        Y_pairs = load_rows(image_captions.language_file(pairs_dataset, model_y).path, pairs)
print(f"X: {tuple(X.shape)} images, Y: {tuple(Y.shape)} captions, {0 if X_pairs is None else len(X_pairs)} known pairs")
if source.value == "paper embeddings" and dataset_x == dataset_y:
    print("first items of X:", indices_x[:5].tolist(), " first items of Y:", indices_y[:5].tolist(), " shared items: 0")

## 4. Our aligner: Wasserstein Procrustes with a geometric initialization

We look for a semi-orthogonal map $W \in \mathrm{St}(d_X, d_Y)$ and a correspondence $T$ between the two sets that solve
$$\max_{W,\, T} \operatorname{Tr}(X W Y^\top T^\top).$$
For a fixed $T$ the best $W$ is orthogonal Procrustes, $W = \mathrm{polar}(X^\top T Y)$, and for a fixed $W$ the best $T$ is a linear assignment. Alternating the two needs a good start, which the **geometric initialization** provides by matching cluster centers of both spaces.

The whole method has **four hyperparameters**, the same for every experiment of the paper:

In [ ]:
C = 30  # number of clusters
S = 30  # number of restarts of the initialization
b = 10_000  # batch size, used by every stage
R = 100  # number of refinement iterations

Both spaces are centered with the mean of their own training set and every row is scaled to unit length.

In [ ]:
torch.set_num_threads(4)  # long sums over many rows depend on the thread count in their last bits


def center_and_normalize(samples, mean):
    centered = samples - mean
    return centered / centered.norm(dim=-1, keepdim=True).clamp(min=1e-10)


def polar(matrix):
    """The orthogonal Procrustes solution polar(N) = U Vᵀ, where N = U Σ Vᵀ."""
    U, _, Vt = torch.linalg.svd(matrix, full_matrices=False)
    return U @ Vt

### Step 1: geometric initialization (Algorithm 2)

For each of the $S$ restarts we draw $b$ samples of each space, cluster them with k-means into $C$ centers $A_s$ and $B_s$, and match the centers by the permutation that maximizes their linear CKA,
$$P_s \in \arg\max_{P} \mathrm{CKA}(A_s, P B_s) = \arg\max_{P} \operatorname{Tr}(\bar K_A P \bar K_B P^\top), \qquad \bar K = H K H,$$
a quadratic assignment problem (Lemma 1), which we solve with MPOpt. Known pairs join both sets and only add a linear term (Lemma 3).
Each matched clustering is a rank-$C$ transport plan with cross-covariance $\frac{1}{C} A_s^\top P_s B_s$, and their average $M$ is the coarse correspondence.

In [ ]:
def kmeans(samples, seed):
    with threadpool_limits(1):  # with several threads, scikit-learn's k-means is not deterministic
        centers = KMeans(n_clusters=C, random_state=seed).fit(samples.numpy()).cluster_centers_
    return torch.as_tensor(centers)


def centered_kernel(samples):
    """H K H with K = samples samplesᵀ and H = I - 11ᵀ/n."""
    K = samples @ samples.T
    return K - K.mean(dim=0, keepdim=True) - K.mean(dim=1, keepdim=True) + K.mean()


def match_clusters(A, B, X_pairs, Y_pairs, seed):
    """argmax_P CKA(vstack(A, X̂), vstack(P B, Ŷ)), solved with MPOpt.

    Without pairs, this is the QAP max_P Tr(K̄_A P K̄_B Pᵀ) (Lemma 1). Pairs add the linear term 2 Tr(K̃_12 L̃_12ᵀ Pᵀ) (Lemma 3).
    """
    if X_pairs is None:
        linear = None
        K_A, K_B = centered_kernel(A), centered_kernel(B)
    else:  # center the kernels of vstack(A, X̂) and vstack(B, Ŷ) jointly and keep the center-center and center-pair blocks
        w = 1.0 / (C + len(X_pairs))
        A_c, X_c = A - (w * A.sum(0) + w * X_pairs.sum(0)), X_pairs - (w * A.sum(0) + w * X_pairs.sum(0))
        B_c, Y_c = B - (w * B.sum(0) + w * Y_pairs.sum(0)), Y_pairs - (w * B.sum(0) + w * Y_pairs.sum(0))
        K_A, K_B = A_c @ A_c.T, B_c @ B_c.T
        linear = (-2.0 * (A_c @ X_c.T)) @ (B_c @ Y_c.T).T
    return MPOptQAPSolver().solve(-K_A, K_B, linear, seed)  # MPOpt minimizes, so the first kernel is negated


def geometric_initialization(X, Y, X_pairs, Y_pairs, randomness):
    cross_covariances = []
    for s in range(S):
        X_B = X[randomness.subset(len(X), b)] if len(X) > b else X  # random batches of b samples
        Y_B = Y[randomness.subset(len(Y), b)] if len(Y) > b else Y
        A = kmeans(X_B, randomness.kmeans_seed())
        B = kmeans(Y_B, randomness.kmeans_seed())
        P = match_clusters(A, B, X_pairs, Y_pairs, randomness.solver_seed())  # B[P] is matched to A
        cross_covariances.append((A * torch.full((C,), 1 / C)[:, None]).T @ B[P])  # (1/C) A_sᵀ P_s B_s
    return torch.stack(cross_covariances).mean(dim=0)  # M = 1/(SC) Σ_s A_sᵀ P_s B_s

### Step 2: read-out

One conditional-gradient step from the averaged plan turns $M$ into sample pairs: we assign on the scores $X M Y^\top$.
A full $n \times m$ assignment is too large, so both sets are shuffled once and the $i$-th block of $b$ rows of $X$ is assigned to the $i$-th block of $Y$ (Jonker-Volgenant).
Procrustes on these $\min(n, m)$ pseudo-pairs gives the first map. Known pairs are weighted so that they count as much as all pseudo-pairs together.

In [ ]:
def hungarian(scores):
    rows, columns = linear_sum_assignment(-scores.double().numpy())  # maximizes the total score
    return torch.as_tensor(rows), torch.as_tensor(columns)


def procrustes(X_matched, Y_matched, X_pairs, Y_pairs):
    """W = polar(Xᵀ T Y + (k/p) X̂ᵀ Ŷ), so the p known pairs weigh as much as the k pseudo-pairs together."""
    cross_covariance = X_matched.T @ Y_matched
    if X_pairs is not None:
        cross_covariance = cross_covariance + (len(X_matched) / len(X_pairs) * X_pairs.T) @ Y_pairs
    return polar(cross_covariance)


def readout(X, Y, M, X_pairs, Y_pairs, randomness):
    scores_X = X @ M  # scores X M Yᵀ of one conditional-gradient step from the averaged plan
    order_X, order_Y = randomness.permutation(len(X)), randomness.permutation(len(Y))  # shuffle both sets once
    X_matched, Y_matched = [], []
    for start in range(0, min(len(X), len(Y)), b):  # assign block i of X to block i of Y
        block_X, block_Y = order_X[start : start + b], order_Y[start : start + b]
        rows, columns = hungarian(scores_X[block_X] @ Y[block_Y].T)
        X_matched.append(X[block_X[rows]])
        Y_matched.append(Y[block_Y[columns]])
    return procrustes(torch.cat(X_matched), torch.cat(Y_matched), X_pairs, Y_pairs)

### Step 3: refinement (Algorithm 1)

Alternate, on random batches of $b$ samples, a linear assignment $T = \mathrm{hungarian}(X_B W Y_B^\top)$ and a Procrustes update.

In [ ]:
def refine(X, Y, W, X_pairs, Y_pairs, randomness):
    for r in range(R):
        X_B = X[randomness.subset(len(X), b)]  # random batches of b samples
        Y_B = Y[randomness.subset(len(Y), b)]
        rows, columns = hungarian((X_B @ W) @ Y_B.T)  # T = hungarian(X_B W Y_Bᵀ)
        W = procrustes(X_B[rows], Y_B[columns], X_pairs, Y_pairs)  # W = polar(X_Bᵀ T Y_B + b/p X̂ᵀŶ)
    return W

### All together

In [ ]:
def wasserstein_procrustes(X, Y, X_pairs, Y_pairs, randomness):
    """Algorithm 1: returns the means of the two training sets and the map W."""
    mean_X, mean_Y = X.mean(dim=0, keepdim=True), Y.mean(dim=0, keepdim=True)
    X, Y = center_and_normalize(X, mean_X), center_and_normalize(Y, mean_Y)
    if X_pairs is not None:
        X_pairs, Y_pairs = center_and_normalize(X_pairs, mean_X), center_and_normalize(Y_pairs, mean_Y)
    M = geometric_initialization(X, Y, X_pairs, Y_pairs, randomness)
    W = readout(X, Y, M, X_pairs, Y_pairs, randomness)
    W = refine(X, Y, W, X_pairs, Y_pairs, randomness)
    return mean_X, mean_Y, W

In [ ]:
mean_X, mean_Y, W = wasserstein_procrustes(X, Y, X_pairs, Y_pairs, randomness)
print("W:", tuple(W.shape), " semi-orthogonal:", torch.allclose(W @ W.T, torch.eye(W.shape[0]), atol=1e-4))

## 5. Validation

We score the aligner on the paired MS COCO validation split with **FOSCTTM**, the fraction of captions that are closer to an image than its true caption (0 = perfect, 0.5 = chance), with ties counting half.

In [ ]:
def similarity(queries_x, keys_y):
    # the aligner: inner products (= cosine similarities) of the mapped images and the captions
    return (center_and_normalize(queries_x, mean_X) @ W) @ center_and_normalize(keys_y, mean_Y).T


def foscttm(similarity, val_x, val_y, randomness, block_size=4096):
    n = len(val_x)
    order_x, order_y = randomness.permutation(n), randomness.permutation(n)
    shuffled_x, shuffled_y = val_x[order_x], val_y[order_y]  # what the aligner sees
    partner = torch.argsort(order_y)[order_x]  # known only here: shuffled_y[partner[i]] belongs to shuffled_x[i]
    closer = torch.empty(n, dtype=torch.float64)
    for start in range(0, n, block_size):
        scores = similarity(shuffled_x[start : start + block_size], shuffled_y).double()
        true_scores = scores[torch.arange(len(scores)), partner[start : start + block_size]][:, None]
        # keys scoring higher than the true partner, and half of the tied keys (the partner itself excluded)
        num_higher, num_tied = (scores > true_scores).sum(1).double(), (scores == true_scores).sum(1).double() - 1
        closer[order_x[start : start + block_size]] = num_higher + 0.5 * num_tied
    return (closer / (n - 1)).mean().item()

In [ ]:
evaluation_randomness = Randomness(seed.value)
if source.value == "own embeddings":
    val_x, val_y = load_rows(own_val_x.value), load_rows(own_val_y.value)
else:
    val_path_x = image_captions.vision_file(image_captions.VALIDATION_DATASET, model_x).path
    val_path_y = image_captions.language_file(image_captions.VALIDATION_DATASET, model_y).path
    val_indices = evaluation_randomness.permutation(num_rows(val_path_x))
    val_x, val_y = load_rows(val_path_x, val_indices), load_rows(val_path_y, val_indices)
result = foscttm(similarity, val_x, val_y, evaluation_randomness)
print(f"FOSCTTM of our aligner: {result:.4f} on {len(val_x)} validation pairs")

## 6. Zero-shot classification

Map every image into the language space and pick the most similar class prompt. The class prompts and the images are shuffled independently as well.

In [ ]:
accuracies = {}
for dataset in [] if source.value == "own embeddings" else classification.value:
    images = load_rows(image_captions.vision_file(dataset, model_x).path)
    prompts = load_rows(image_captions.language_file(dataset, model_y).path)
    labels = torch.load(os.path.join(os.environ["UNPAIRED_ROSETTA_ROOT"], image_captions.label_file(dataset)))
    order_images, order_classes = evaluation_randomness.permutation(len(images)), evaluation_randomness.permutation(len(prompts))
    top5 = order_classes[similarity(images[order_images], prompts[order_classes]).topk(5, dim=1).indices]
    k = 1 if image_captions.CLASSIFICATION_DATASETS[dataset] == "top1" else 5
    accuracies[dataset] = (top5[:, :k] == labels[order_images][:, None]).any(dim=1).float().mean().item()
    print(f"{dataset} top-{k} accuracy: {100 * accuracies[dataset]:.1f}%")